# Database Exploration
This notebook provides a quick way to inspect the schema and head of the RBA and FP Stalker databases.

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# Paths to databases relative to this notebook
RBA_DB_PATH = Path("./data/raw/rba.duckdb")
FP_STALKER_DB_PATH = Path("./data/raw/fp_stalker.duckdb")

print(f"RBA Database exists: {RBA_DB_PATH.exists()}")
print(f"FP Stalker Database exists: {FP_STALKER_DB_PATH.exists()}")

## 1. Inspect RBA Database

In [ ]:
if RBA_DB_PATH.exists():
    with duckdb.connect(str(RBA_DB_PATH)) as conn:
        # Show tables
        print("--- Tables in RBA ---")
        display(conn.execute("PRAGMA show_tables").df())
        
        # Describe the imported_data table
        print("\n--- Schema of imported_data ---")
        display(conn.execute("DESCRIBE imported_data").df())
        
        # Show first 5 rows
        print("\n--- First 5 rows ---")
        display(conn.execute("SELECT * FROM imported_data LIMIT 5").df())
else:
    print("RBA database file not found. Please run fetch_data.py first.")

In [ ]:
rba_conn = duckdb.connect(str(RBA_DB_PATH))

display(
    rba_conn.execute(
        """
        SELECT COUNT(DISTINCT "User ID") AS distinct_user_count
        FROM imported_data
        WHERE "Is Account Takeover" IS TRUE
        """
    ).df()
)



display(
    rba_conn.execute(
        """
        SELECT COUNT(DISTINCT "User ID") AS distinct_user_count
        FROM imported_data
        WHERE "Is Attack IP" IS TRUE
        """
    ).df()
)


display(
    rba_conn.execute(
        """
        SELECT COUNT(DISTINCT "User ID") AS distinct_user_count
        FROM imported_data
        """
    ).df()
)

display(
    rba_conn.execute(
        """
SELECT * FROM imported_data
WHERE "User ID" IN  (
    SELECT DISTINCT("User ID")
    FROM imported_data
    WHERE "Is Account Takeover" IS TRUE
)
""").df().head()
)

In [ ]:
# rba_conn.execute("ATTACH './demo/trunc_wiefling_rba.duckdb' AS rba_db")
# rba_conn.execute("""
#     CREATE TABLE rba_db.imported_data AS (
#         SELECT * FROM imported_data
#             WHERE "User ID" IN  (
#                 SELECT DISTINCT("User ID")
#                 FROM imported_data
#                 WHERE "Is Account Takeover" IS TRUE
#             )
#         )
# """)
# rba_conn.execute("DETACH rba_db")
# rba_conn.close()

In [ ]:
rba_conn2 = duckdb.connect('./demo/trunc_wiefling_rba.duckdb')
print(rba_conn2.execute("SELECT * FROM imported_data").df().shape)
display(rba_conn2.execute("SELECT * FROM imported_data").df().head())

rba_conn2.execute(
        """
SELECT COUNT(DISTINCT "User ID") AS distinct_user_count
FROM imported_data
WHERE "Is Account Takeover" IS TRUE
""").df()   

In [ ]:
rba_conn2.close()

## 2. Inspect FP Stalker Database

In [ ]:
if FP_STALKER_DB_PATH.exists():
    with duckdb.connect(str(FP_STALKER_DB_PATH)) as conn:
        # Show tables
        print("--- Tables in FP Stalker ---")
        display(conn.execute("PRAGMA show_tables").df())
        
        # Describe the imported_data table
        print("\n--- Schema of imported_data ---")
        display(conn.execute("DESCRIBE imported_data").df())
        
        # Show first 5 rows
        print("\n--- First 5 rows ---")
        display(conn.execute("SELECT * FROM imported_data LIMIT 5").df())
else:
    print("FP Stalker database file not found. Please run fetch_data.py first.")

In [ ]:
fpconn = duckdb.connect(str(FP_STALKER_DB_PATH))
df = fpconn.execute("SELECT * FROM imported_data").df()

print(df.shape)
df

In [ ]:
distinct_ids = df['id'].unique()
print(f"Number of distinct IDs: {len(distinct_ids)}")

distinct_ids = df['id'].unique()
print(f"Number of distinct IDs: {len(distinct_ids)}")

grouped_df = (
    df.groupby("id", as_index=False)
    .agg(
        earliest_creation_date=("creationDate", lambda s: s.dropna().min() if not s.dropna().empty else pd.NaT),
        latest_creation_date=("creationDate", lambda s: s.dropna().max() if not s.dropna().empty else pd.NaT),
        earliest_end_date=("endDate", lambda s: s.dropna().min() if not s.dropna().empty else pd.NaT),
        last_end_date=("endDate", lambda s: s.dropna().max() if not s.dropna().empty else pd.NaT),
    )
)
grouped_df["start"] = grouped_df["earliest_creation_date"]
grouped_df["end"] = grouped_df["last_end_date"].fillna(grouped_df["latest_creation_date"])
grouped_df = grouped_df.drop(columns=["earliest_creation_date", "latest_creation_date", "earliest_end_date", "last_end_date"])
grouped_df['duration'] = (grouped_df['end'] - grouped_df['start']).dt.total_seconds() / 86400
display(grouped_df.head())

import random
sample_ids = random.sample(list(distinct_ids), 100)

In [ ]:
grouped_df

In [ ]:
june16_df = grouped_df[(grouped_df["start"] < pd.Timestamp("2016-06-01")) & (grouped_df["end"] >= pd.Timestamp("2016-07-01"))].copy()

june16_ids = june16_df['id'].tolist()
len(june16_ids)

In [ ]:
fp_df_trunc = df[df['id'].isin(june16_ids)]

cols = ['counter', 'id', 'creationDate', 'updateDate', 'endDate', 
        'userAgentHttp', 'osDetailed', 'browserDetailed', 'browserVersion', 
        'platformJS', 'resolutionJS']

fp_df_trunc = fp_df_trunc[cols]

In [ ]:
out_path = Path("trunc_fp_stalker.duckdb")

with duckdb.connect(str(out_path)) as out_conn:
    out_conn.execute("CREATE OR REPLACE TABLE imported_data AS SELECT * FROM fp_df_trunc")

print(f"Wrote {len(fp_df_trunc)} rows to {out_path}")